# Verifica: PDT e RuleCard costruiscono le stesse coppie?

Sia il PDT sia RuleCard, per imparare una distanza tra candidati, costruiscono una tabella di **coppie**. Ogni riga della tabella ha due parti:

- **rappresentazione della coppia** (dei numeri: ad es. la differenza assoluta `|a - b|` feature per feature);
- **target da imparare**: la distanza euclidea al quadrato fra i valori `z` che RTR passa, `(z_a - z_b)^2`. Cosa siano quei `z` lo decide `dist_objective`: le feature con `"dist"` (il default), le etichette con `"y"`, i residui con `"residuals"`.

La domanda è: queste due tabelle coincidono? Se PDT e RuleCard partissero da tabelle diverse, una differenza nei risultati potrebbe venire da lì e non dal modello, e il confronto non sarebbe ad armi pari.

Il test giusto **non** è "gli output sono identici riga per riga" (le coppie vengono pescate a caso con generatori diversi, quindi non lo sono). Verifichiamo invece due invarianti:

1. **a parità di indici di coppia**, la rappresentazione del PDT (`_generate_pairwise_x_dataset`) e quella di RuleCard (`_gen_pairs`, via `_build_pairs_numba` + `_apply_pair_feature_view`) escono numericamente uguali;
2. il **target** che RuleCard riceve in `mode='precomputed'` coincide con `euclidean_distances(z, squared=True)` del PDT.

Se entrambi valgono, i due modelli imparano la stessa funzione obiettivo.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ruletreerank").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from sklearn.metrics import euclidean_distances

from ruletreerank.pdt import PairwiseDistanceTree
from ruletreerank.rulecard_pdt import RuleCardPairwiseDistance
from PairwiseRuleCard.PairwiseRuleCardGAM import _apply_pair_feature_view
from PairwiseRuleCard.utils import _build_pairs_numba

# esempio piccolo e leggibile: pochi candidati, poche feature
rng = np.random.default_rng(0)
N, D = 5, 4
X = np.round(rng.normal(size=(N, D)), 3)
# residui del primo stadio: è su questi che si calcola il target della distanza
z = np.round(rng.normal(size=N), 3)

print("Candidati X (righe = candidati, colonne = feature):")
print(pd.DataFrame(X, index=[f"c{i}" for i in range(N)], columns=[f"f{j}" for j in range(D)]))
print("\nResidui z (uno per candidato):")
print(pd.Series(z, index=[f"c{i}" for i in range(N)]))

Candidati X (righe = candidati, colonne = feature):
       f0     f1     f2     f3
c0  0.126 -0.132  0.640  0.105
c1 -0.536  0.362  1.304  0.947
c2 -0.704 -1.265 -0.623  0.041
c3 -2.325 -0.219 -1.246 -0.732
c4 -0.544 -0.316  0.412  1.043

Residui z (uno per candidato):
c0   -0.129
c1    1.366
c2   -0.665
c3    0.352
c4    0.903
dtype: float64


C:\Users\manzo\Desktop\fork\RuleTreeRank-DS2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Invariante 1 - la rappresentazione delle coppie

Fissiamo lo **stesso insieme di coppie** su entrambi i lati: consideriamo tutti i candidati, quindi tutte le `N*N` coppie in ordine cartesiano `(i, j)`. Confrontiamo la rappresentazione con differenza assoluta `|a - b|` (l'unica in comune tra PDT e RuleCard).

In [2]:
# stessi flag di rappresentazione: solo differenza assoluta |a - b|
pdt = PairwiseDistanceTree(feature_concat=False, feature_diff=True, feature_sq_diff=False)

mask = np.ones(N, dtype=bool)  # tutti i candidati -> tutte le N*N coppie

# lato PDT
x_pdt = pdt._generate_pairwise_x_dataset(X, mask, mask)

# lato RuleCard: costruisce le coppie concatenate e poi ne ricava |a - b|
pairs_concat = _build_pairs_numba(X, X)
x_rc = _apply_pair_feature_view(pairs_concat, D, use_pairwise=False, use_difference=True)

print("shape PDT:", x_pdt.shape, "| shape RuleCard:", x_rc.shape)

idx = [(i, j) for i in range(N) for j in range(N)]
head = pd.DataFrame({
    "coppia": [f"c{i}-c{j}" for i, j in idx][:6],
    "|a-b| PDT": [np.round(r, 3).tolist() for r in x_pdt[:6]],
    "|a-b| RuleCard": [np.round(r, 3).tolist() for r in x_rc[:6]],
})
print(head.to_string(index=False))

print("\nRappresentazioni identiche:", np.allclose(x_pdt, x_rc))

shape PDT: (25, 4) | shape RuleCard: (25, 4)
coppia                    |a-b| PDT               |a-b| RuleCard
 c0-c0         [0.0, 0.0, 0.0, 0.0]         [0.0, 0.0, 0.0, 0.0]
 c0-c1 [0.662, 0.494, 0.664, 0.842] [0.662, 0.494, 0.664, 0.842]
 c0-c2  [0.83, 1.133, 1.263, 0.064]  [0.83, 1.133, 1.263, 0.064]
 c0-c3 [2.451, 0.087, 1.886, 0.837] [2.451, 0.087, 1.886, 0.837]
 c0-c4  [0.67, 0.184, 0.228, 0.938]  [0.67, 0.184, 0.228, 0.938]
 c1-c0 [0.662, 0.494, 0.664, 0.842] [0.662, 0.494, 0.664, 0.842]

Rappresentazioni identiche: True


Per completezza, anche la rappresentazione con concatenazione `[a, b]` coincide.

In [3]:
pdt_cat = PairwiseDistanceTree(feature_concat=True, feature_diff=False, feature_sq_diff=False)
x_pdt_cat = pdt_cat._generate_pairwise_x_dataset(X, mask, mask)
x_rc_cat = _apply_pair_feature_view(pairs_concat, D, use_pairwise=True, use_difference=False)
print("Concatenazione [a, b] identica:", np.allclose(x_pdt_cat, x_rc_cat))

Concatenazione [a, b] identica: True


## Invariante 2 - il target

Il target è la distanza euclidea al quadrato fra i valori `z`. Il PDT la calcola dentro `fit`; l'adapter la calcola dentro `_pair_target` con la stessa identica formula e la consegna a RuleCard già pronta (`mode='precomputed'`), quindi RuleCard non la ricalcola. Qui l'esempio usa dei residui, ma l'invariante vale per qualunque `z`: è la formula a coincidere, non il particolare contenuto.

In [4]:
# lato PDT: il target calcolato dentro fit è la distanza euclidea^2 fra i valori z
y_pdt = euclidean_distances(z.reshape(-1, 1), z.reshape(-1, 1), squared=True).ravel()

# lato adapter: stessa formula, dentro _pair_target
rc = RuleCardPairwiseDistance(feature_concat=False, feature_diff=True, feature_sq_diff=False)
D_mat = rc._pair_target(X, z, None, mask, N)
y_rc = D_mat.ravel()

tab = pd.DataFrame({
    "coppia": [f"c{i}-c{j}" for i, j in idx][:6],
    "(z_i - z_j)^2 a mano": [round((z[i] - z[j]) ** 2, 4) for i, j in idx][:6],
    "target PDT": np.round(y_pdt[:6], 4),
    "target adapter": np.round(y_rc[:6], 4),
})
print(tab.to_string(index=False))

print("\nTarget identici:", np.allclose(y_pdt, y_rc))

coppia  (z_i - z_j)^2 a mano  bersaglio PDT  bersaglio adapter
 c0-c0                0.0000         0.0000             0.0000
 c0-c1                2.2350         2.2350             2.2350
 c0-c2                0.2873         0.2873             0.2873
 c0-c3                0.2314         0.2314             0.2314
 c0-c4                1.0650         1.0650             1.0650
 c1-c0                2.2350         2.2350             2.2350

Bersagli identici: True


## Il punto centrale

Il motivo per cui il target è identico non è che `_gen_pairs` faccia la stessa cosa di `_generate_pairwise_x_dataset`: è che l'adapter **scavalca** la generazione del target di RuleCard. Calcola lui la distanza euclidea al quadrato sui residui (stessa formula del PDT) e la passa con `mode='precomputed'`. Quindi il target è identico **per costruzione**, indipendentemente da come vengono pescate le coppie.

Restano tre differenze note, nessuna delle quali sposta il confronto:

1. RuleCard non ha la differenza al quadrato `(a - b)^2`; ha solo concatenazione e `|a - b|`. Ma `|a - b|` porta la stessa informazione di ordine, che è ciò che serve al kNN.
2. Le coppie vengono pescate con generatori casuali diversi: anche con lo stesso seed i due lati non estraggono le stesse coppie (vedi cella sotto). Sono comunque campioni non distorti dello stesso insieme, verso lo stesso target.
3. RuleCard tiene da parte una fetta di validazione (15%) per l'early-stopping del boosting; il PDT no. È intrinseco al boosting.

In [5]:
# perché un confronto "coppie identiche riga per riga" fallirebbe (e non è un bug):
# i due lati campionano le coppie con generatori diversi.
seed, sub = 7, 0.6
n = int(round(N * sub))

# campionamento come nel PDT (numpy Generator)
g = np.random.default_rng(seed)
pdt_a = g.choice(N, n, replace=False)
pdt_b = g.choice(N, n, replace=False)

# campionamento come in RuleCard (stato globale legacy)
np.random.seed(seed)
rc_a = np.random.choice(N, n, replace=False)
rc_b = np.random.choice(N, n, replace=False)

print("coppie campionate dal PDT     :", list(zip(pdt_a.tolist(), pdt_b.tolist())))
print("coppie campionate da RuleCard :", list(zip(rc_a.tolist(), rc_b.tolist())))
print("\nInsiemi di coppie diversi -> un confronto riga-per-riga fallirebbe.")
print("Ma per qualsiasi coppia (i, j) il target è lo stesso valore (z_i - z_j)^2,")
print("quindi i due modelli imparano la stessa funzione obiettivo.")

coppie campionate dal PDT     : [(2, 3), (3, 1), (4, 2)]
coppie campionate da RuleCard : [(0, 2), (3, 1), (2, 0)]

Insiemi di coppie diversi -> un confronto riga-per-riga fallirebbe.
Ma per qualsiasi coppia (i, j) il bersaglio e' lo stesso valore (z_i - z_j)^2,
quindi i due modelli imparano la stessa funzione obiettivo.


## Conclusione

- **Rappresentazione**: a parità di coppie, PDT e RuleCard producono le stesse tabelle (`|a - b|` e `[a, b]`).
- **Target**: identico per costruzione (distanza euclidea al quadrato sui residui, passata come `precomputed`).

Le uniche differenze sono la mancanza di `(a - b)^2` in RuleCard, il campionamento casuale delle coppie e lo split di validazione interno al boosting: nessuna cambia la funzione obiettivo. Il confronto RTR vs RTRwRuleCard è quindi ad armi pari.